# ULMFiT on WikiText-103 and IMDb, the best configuration on one GPU

This dialog trains ULMFiT with the tokenizer, token cache, transforms and loaders of `fastai.text.tokenizer`. One process trains one direction on one GPU. `ULMFIT_BACKWARD=1` trains the backward model, which reads every document reversed. The forward and backward classifiers are averaged at the end.

These settings gave 95.45% on the IMDb test set:

| Stage | Setting |
|---|---|
| 1, WikiText-103 | 10 epochs, warm-up, plateau and decay schedule, peak rate 1e-2, `drop_mult` 0.1, batch 256 |
| 2, IMDb | 1 frozen epoch at 5e-2, 10 unfrozen epochs at 1e-2, `drop_mult` 0.3, batch 160 |
| 3, classifier | rate 5e-2 halved at each unfreezing step, label smoothing 0.1, 4 final epochs, `drop_mult` 0.5, batch 64 |

They were found on six GPUs, each with its own batch. `GradientAccumulation` over six batches gives one GPU the same number of optimizer steps and the same effective batch, so the learning rates carry over unchanged. `ULMFIT_SMOKE=1` cuts every epoch to a handful of batches, to check the whole pipeline in minutes.


In [ ]:
import os
from fastai.text.all import *
from fastai.text.tokenizer import *

backward = bool(int(os.environ.get('ULMFIT_BACKWARD', 0)))
smoke = bool(int(os.environ.get('ULMFIT_SMOKE', 0)))
run = 'bwd' if backward else 'fwd'
if smoke: run += '_smoke'
data_dir = Path('data/wt103_exp')
(data_dir / 'models').mkdir(parents=True, exist_ok=True)
n_gpus = 6
MOMS = (0.8, 0.7, 0.8)
short = [ShortEpochCallback(pct=0.002, short_valid=False)] if smoke else []
set_seed(42)
run

/Users/hoa/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


'bwd_smoke'

## Tokenizer

`hf_tokenizer` is the pretrained GPT-2 byte-level BPE with `<eos>`, `<bos>` and `<pad>` added as special tokens. The factories below wrap it in `HfTokenizer`, and `HfTextBlock` adds `Reverse` after it for the backward model: the loaders, the model and the classifier stay the same.


In [ ]:
tok = hf_tokenizer()
tok.get_vocab_size(), tok.token_to_id(PAD_TOK)


(50260, 50259)

## WikiText-103

The raw training split from Hugging Face, assembled into one document per ` = Title = ` heading by `assemble_wiki103`. `tokenize_dataset` builds the cache on the first run and loads it afterwards, returning the tokenizer saved with it.


In [ ]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def assemble_wiki103(ds):
    "Join the lines of WikiText-103 dataset `ds` into one document per ` = Title = ` heading"
    docs, cur = [], []
    for text in ds['text']:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    return [d for d in docs if d.strip()]


In [ ]:
flat, offsets, cache_tok = tokenize_dataset('Salesforce/wikitext', assemble_wiki103, 'wikitext-103-raw-v1', cache_dir='cache/wiki103')
wiki_docs = split_docs(flat, offsets)
len(wiki_docs), len(flat)


(29444, 117920140)

In [ ]:
tok.decode(wiki_docs[0])[:200]

' = Valkyria Chronicles III = \n Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Ja'

## Language model loaders and learner

Stage 1 uses `TextDataLoadersNew.from_dataset`. Stage 2 uses `TextDataLoadersNew.from_folder` over the IMDb files. The split is a seeded permutation, 2% held out for WikiText-103 and 10% for IMDb. `lm_learner` sets the optimizer: Adam with `eps=1e-7`, gradient clipping at 0.1, mixed precision, and gradient accumulation over six batches.


In [ ]:
def lm_learner(dls, drop_mult):
    "AWD-LSTM language model from scratch, with mixed precision, gradient clipping at 0.1 and gradient accumulation over `n_gpus` batches."
    config = awd_lstm_lm_config.copy()
    config['pad_token'] = dls.pad_id
    learn = language_model_learner(dls, AWD_LSTM, config=config, drop_mult=drop_mult, pretrained=False,
        metrics=[accuracy, Perplexity()], path=data_dir, opt_func=partial(Adam, eps=1e-7),
        cbs=[GradientClip(0.1), GradientAccumulation(n_acc=dls.bs * n_gpus)]).to_fp16()
    return learn

## Stage 1: pretrain on WikiText-103

The warm-up, plateau and decay schedule: one epoch of linear warm-up from a tenth of the peak rate, seven epochs at the peak, two epochs of cosine decay to a hundredth. Momentum stays at 0.8. `SaveModelCallback` keeps the epoch with the best validation loss, which is the last one.

In [ ]:
def wsd_scheds(lr, epochs, warm_epochs, decay_epochs, final_div=100):
    "Warm-up, plateau and cosine decay over `epochs`."
    phases = [(warm_epochs, SchedLin(lr / 10, lr)), (epochs - warm_epochs - decay_epochs, SchedNo(lr, lr)), (decay_epochs, SchedCos(lr, lr / final_div))]
    return {'lr': combine_scheds([n / epochs for n, _ in phases], [s for _, s in phases]), 'mom': SchedNo(0.8, 0.8)}

dls_wiki = TextDataLoadersNew.from_dataset('Salesforce/wikitext', assemble_wiki103, 'wikitext-103-raw-v1',
    cache_dir='cache/wiki103', valid_pct=0.02, seed=42, backwards=backward, bs=256)
learn = lm_learner(dls_wiki, drop_mult=0.1)
len(dls_wiki.train), len(dls_wiki.valid)

(6279, 123)

In [ ]:
epochs = 10
learn.fit(epochs, 1e-2, wd=0.1, cbs=[ParamScheduler(wsd_scheds(1e-2, epochs, 1, 2)), SaveModelCallback(monitor='valid_loss', fname=f'{run}_lm_best')] + short)
learn.save(f'{run}_lm')
del learn, dls_wiki

## IMDb

The reviews are text files in `train` and `test` folders. `from_folder` caches them with `tokenize_folder` on the first run: all 100,000 for stage 2, and the labelled 50,000 for stage 3, with the folder names as label and split.


In [ ]:
path = untar_data(URLs.IMDB)


## Stage 2: fine-tune on IMDb

One epoch with only the last layer group trainable at 5e-2, then ten unfrozen epochs at a flat 1e-2, with `drop_mult` 0.3. The encoder of the epoch with the best validation loss is saved.

In [ ]:
dls_imdb = TextDataLoadersNew.from_folder(path, is_lm=True, valid_pct=0.1, seed=42, folders=['train', 'test', 'unsup'], backwards=backward, bs=160)
learn = lm_learner(dls_imdb, drop_mult=0.3)
learn.load(f'{run}_lm')
learn.freeze()
learn.fit_one_cycle(1, 5e-2, wd=0.1, moms=MOMS, cbs=short)
learn.unfreeze()
learn.fit_one_cycle(10, 1e-2, wd=0.1, moms=MOMS, cbs=[SaveModelCallback(monitor='valid_loss', fname=f'{run}_ft_best')] + short)
learn.save_encoder(f'{run}_ft_enc')
del learn, dls_imdb

## Classifier loaders

`from_folder` with `valid='test'` splits by folder. `HfTextBlock` without `is_lm` pads with `Pad_Chunk`, which puts whole padding chunks before the text and the remainder after it, for both directions.


In [ ]:
dls_clas = TextDataLoadersNew.from_folder(path, valid='test', backwards=backward, bs=64)
len(dls_clas.train_ds), len(dls_clas.valid_ds), dls_clas.vocab[1]


(25000, 25000, ['neg', 'pos'])

## Stage 3: classifier

Label smoothing 0.1 and a halving schedule: one epoch on the head at 5e-2, one epoch each after unfreezing the next two layer groups at half the rate, then four epochs fully unfrozen at a fifth of that. The last epoch is kept. The test probabilities are saved for the ensemble.

In [ ]:
config = awd_lstm_clas_config.copy()
config['pad_token'] = dls_clas.pad_id
learn = text_classifier_learner(dls_clas, AWD_LSTM, seq_len=72, config=config, drop_mult=0.5, pretrained=False,
    metrics=accuracy, path=data_dir, opt_func=partial(Adam, eps=1e-7), cbs=GradientClip(0.1),
    loss_func=LabelSmoothingCrossEntropyFlat(eps=0.1)).to_fp16()
learn.load_encoder(f'{run}_ft_enc')
lr = 5e-2
learn.fit_one_cycle(1, lr, wd=0.1, moms=MOMS, cbs=short)
for k in (-2, -3):
    learn.freeze_to(k)
    lr /= 2
    learn.fit_one_cycle(1, slice(lr / 2.6**4, lr), wd=0.1, moms=MOMS, cbs=short)
lr /= 5
learn.unfreeze()
learn.fit_one_cycle(4, slice(lr / 2.6**4, lr), wd=0.1, moms=MOMS, cbs=short)
learn.save(f'{run}_clas')
probs, targs = learn.get_preds(dl=dls_clas.valid)
torch.save({'probs': probs, 'targs': targs}, data_dir / 'models' / f'{run}_probs.pt')
print(f'{run} test accuracy: {(probs.argmax(1) == targs).float().mean():.4f}')

## Ensemble

The average of the two directions' test probabilities, once both runs have finished. `SortedDL` orders the test set the same way in both, and the targets are compared to make sure.

In [ ]:
names = [f'{d}{"_smoke" if smoke else ""}' for d in ('fwd', 'bwd')]
paths = [data_dir / 'models' / f'{nm}_probs.pt' for nm in names]
if all(p.exists() for p in paths):
    saved = [torch.load(p) for p in paths]
    assert torch.equal(saved[0]['targs'], saved[1]['targs'])
    accs = [(s['probs'].argmax(1) == s['targs']).float().mean().item() for s in saved]
    ens = (((saved[0]['probs'] + saved[1]['probs']) / 2).argmax(1) == saved[0]['targs']).float().mean().item()
    print(f'forward {accs[0]:.4f}  backward {accs[1]:.4f}  ensemble {ens:.4f}')